In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
# prepare inputs and compute Attention
inputs = tokenizer.encode("I am going to Chennai", return_tensors='pt')
outputs = model(inputs)
attention = outputs[-1]  # Output includes attention weights when output_attentions=True
tokens = tokenizer.convert_ids_to_tokens(inputs[0])
print(tokens)
tokens_clean = [t.replace('##', '') for t in tokens]


['[CLS]', 'i', 'am', 'going', 'to', 'chennai', '[SEP]']


In [ ]:
import torch
# Build dictionary and integer sequence
unique_tokens = sorted(set(tokens_clean))
dc = {s: i for i, s in enumerate(unique_tokens)}
sentence_int = torch.tensor([dc[t] for t in tokens_clean])

**Input Embedding**

In [ ]:
# Create embedding layer and embed
embed = torch.nn.Embedding(len(dc), 16)
embedded_sentence = embed(sentence_int).detach()

In [ ]:
# Output
print("Tokens:", tokens_clean)
print("Sentence indices:", sentence_int)
print("Embedded shape:", embedded_sentence.shape)
print("Embedded values:", embedded_sentence)

Tokens: ['[CLS]', 'i', 'am', 'going', 'to', 'chennai', '[SEP]']
Sentence indices: tensor([0, 5, 2, 4, 6, 3, 1])
Embedded shape: torch.Size([7, 16])
Embedded values: tensor([[-0.2001, -2.0945, -0.6927, -0.2875,  1.1033, -1.8771,  0.3250, -1.7284,
         -1.0401, -1.6582, -1.1510,  0.1113,  0.8810,  0.4492, -0.4923, -0.2311],
        [-0.4823, -0.2108,  0.8710,  0.4849,  0.6701, -1.4694,  0.9754, -0.5541,
         -0.5175, -2.4874, -0.9719, -0.1628,  0.0349, -0.0050,  2.1976,  1.6585],
        [-0.3057, -0.6132,  0.5542,  2.4321, -0.2197,  0.4035, -1.3100,  0.0123,
         -1.0247, -0.1438, -0.2873, -1.2557, -0.0046,  1.0525,  0.9524, -0.7223],
        [-0.3162,  1.2700, -0.0903, -0.8620,  1.3937, -0.7313,  0.2600,  0.6740,
          0.4362,  0.1359, -1.0968, -0.6041,  1.4266, -1.2359,  0.2372, -0.5583],
        [ 0.0387,  0.4220,  1.7113,  1.0220, -0.6876,  0.3424,  0.8320, -1.7906,
         -0.0903, -0.1334, -0.7172,  0.2760,  1.5272,  0.2303, -0.6852,  0.8131],
        [ 1.8993, -0

**Defining the Weight Matrices(Q,K,V)**

In [ ]:
torch.manual_seed(123)

d = embedded_sentence.shape[1]

d_q, d_k, d_v = 24, 24, 28

W_query = torch.nn.Parameter(torch.rand(d_q, d))
W_key = torch.nn.Parameter(torch.rand(d_k, d))
W_value = torch.nn.Parameter(torch.rand(d_v, d))
print(W_query)
print(W_key)
print(W_value)

Parameter containing:
tensor([[0.2961, 0.5166, 0.2517, 0.6886, 0.0740, 0.8665, 0.1366, 0.1025, 0.1841,
         0.7264, 0.3153, 0.6871, 0.0756, 0.1966, 0.3164, 0.4017],
        [0.1186, 0.8274, 0.3821, 0.6605, 0.8536, 0.5932, 0.6367, 0.9826, 0.2745,
         0.6584, 0.2775, 0.8573, 0.8993, 0.0390, 0.9268, 0.7388],
        [0.7179, 0.7058, 0.9156, 0.4340, 0.0772, 0.3565, 0.1479, 0.5331, 0.4066,
         0.2318, 0.4545, 0.9737, 0.4606, 0.5159, 0.4220, 0.5786],
        [0.9455, 0.8057, 0.6775, 0.6087, 0.6179, 0.6932, 0.4354, 0.0353, 0.1908,
         0.9268, 0.5299, 0.0950, 0.5789, 0.9131, 0.0275, 0.1634],
        [0.3009, 0.5201, 0.3834, 0.4451, 0.0126, 0.7341, 0.9389, 0.8056, 0.1459,
         0.0969, 0.7076, 0.5112, 0.7050, 0.0114, 0.4702, 0.8526],
        [0.7320, 0.5183, 0.5983, 0.4527, 0.2251, 0.3111, 0.1955, 0.9153, 0.7751,
         0.6749, 0.1166, 0.8858, 0.6568, 0.8459, 0.3033, 0.6060],
        [0.9882, 0.8363, 0.9010, 0.3950, 0.8809, 0.1084, 0.5432, 0.2185, 0.3834,
         0.3720

**Computing the unnormalized Attention Weights**

In [ ]:
x_2 = embedded_sentence[1]
query_2 = W_query.matmul(x_2)
key_2 = W_key.matmul(x_2)
value_2 = W_value.matmul(x_2)

print(query_2.shape)
print(key_2.shape)
print(value_2.shape)
torch.Size([24])
torch.Size([24])
torch.Size([28])

torch.Size([24])
torch.Size([24])
torch.Size([28])


torch.Size([28])

In [ ]:
keys = W_key.matmul(embedded_sentence.T).T
values = W_value.matmul(embedded_sentence.T).T

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([7, 24])
values.shape: torch.Size([7, 28])


In [ ]:
# compute the ω  values for all input tokens

omega_2 = query_2.matmul(keys.T)
print(omega_2)

tensor([ 13.4246,  -5.6719,  11.3628, -12.2053, -13.7265,  -5.3347, -15.1549],
       grad_fn=<SqueezeBackward4>)


**Computing the Attention Scores- Softmax function**

In [ ]:
import torch.nn.functional as F

attention_weights_2 = F.softmax(omega_2 / d_k**0.5, dim=0)
print(attention_weights_2)

tensor([0.5846, 0.0119, 0.3838, 0.0031, 0.0023, 0.0127, 0.0017],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
#compute the context vector z(2),which is an attention-weighted version of our original query input x(2)including all the other input elements as its context via the attention weights:

context_vector_2 = attention_weights_2.matmul(values)

print(context_vector_2.shape)
print(context_vector_2)

torch.Size([28])
tensor([ 1.7246,  1.6803,  3.5575,  1.6885,  2.5448,  2.3825, -0.3550,  2.3866,
         1.7087,  1.7806,  1.8056,  2.6402,  2.4630,  3.9425,  1.8225,  1.7101,
         0.8429,  0.2109,  0.5776,  1.3559,  2.1572,  1.2847,  1.5722,  0.9678,
         0.8267,  2.9516,  1.0900,  2.9980], grad_fn=<SqueezeBackward4>)
